In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install opencv-python mediapipe numpy pandas matplotlib tqdm


INFO: pip is looking at multiple versions of mediapipe to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 21.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.


In [ ]:

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

# ---- USER: set your video path ----
VIDEO_PATH = "/content/drive/MyDrive/ESRGAN/test.mp4"
OUT_DIR = "/content/gaze_debug"
os.makedirs(OUT_DIR, exist_ok=True)
IMAGES_DIR = os.path.join(OUT_DIR, "frames")
os.makedirs(IMAGES_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUT_DIR, "predictions.csv")

# -----------------------------
# FaceMesh initialization (reused)
# -----------------------------
mp_face = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# -----------------------------
# Eye landmark indices
# -----------------------------
LEFT_EYE_LMS  = [33, 133, 160, 159, 158, 157, 173]
LEFT_IRIS_LMS = [468, 469, 470, 471]

RIGHT_EYE_LMS  = [362, 263, 387, 386, 385, 384, 398]
RIGHT_IRIS_LMS = [473, 474, 475, 476]

In [ ]:
# -----------------------------
# Helper crop functions (kept)
# -----------------------------
def crop_single_eye(frame, eye_landmarks, iris_landmarks, lm_list):
    h, w, _ = frame.shape
    indices = eye_landmarks + iris_landmarks
    xs = [lm_list[i].x * w for i in indices]
    ys = [lm_list[i].y * h for i in indices]
    x1, x2 = int(max(min(xs)-5,0)), int(min(max(xs)+5,w))
    y1, y2 = int(max(min(ys)-5,0)), int(min(max(ys)+5,h))
    eye_patch = frame[y1:y2, x1:x2]
    return eye_patch, (x1, x2, y1, y2)

def crop_both_eyes(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = mp_face.process(rgb)
    if not results.multi_face_landmarks:
        return None, None, (None, None)
    lm = results.multi_face_landmarks[0].landmark
    left_eye, left_box = crop_single_eye(frame, LEFT_EYE_LMS, LEFT_IRIS_LMS, lm)
    right_eye, right_box = crop_single_eye(frame, RIGHT_EYE_LMS, RIGHT_IRIS_LMS, lm)
    return left_eye, right_eye, (left_box, right_box)

# -----------------------------
# Helpers: iris centroid, blink score, EMA
# -----------------------------
def iris_centroid_and_ratio(landmarks, iris_indices, eye_box, frame_w, frame_h):
    x1, x2, y1, y2 = eye_box
    w_box = max(x2 - x1, 1)
    h_box = max(y2 - y1, 1)
    xs = [landmarks[i].x * frame_w for i in iris_indices]
    ys = [landmarks[i].y * frame_h for i in iris_indices]
    cx = float(np.mean(xs))
    cy = float(np.mean(ys))
    ratio_x = (cx - x1) / w_box
    ratio_y = (cy - y1) / h_box
    return (cx, cy), (ratio_x, ratio_y)

def eye_open_score(landmarks, eye_landmarks, frame_h):
    ys = [landmarks[i].y * frame_h for i in eye_landmarks]
    return float(max(ys) - min(ys))

class EMA:
    def __init__(self, alpha=0.6):
        self.alpha = alpha
        self.val = None
    def update(self, x):
        if x is None:
            return self.val
        if self.val is None:
            self.val = x
        else:
            self.val = self.alpha * x + (1-self.alpha) * self.val
        return self.val


# --- reprojection error and axes drawing ---
def _reprojection_error(model_points, image_points, rvec, tvec, camera_matrix, dist_coeffs):
    proj, _ = cv2.projectPoints(model_points, rvec, tvec, camera_matrix, dist_coeffs)
    proj = proj.reshape(-1, 2)
    err = np.linalg.norm(proj - image_points, axis=1)
    return float(np.mean(err))

def draw_axes(frame, rvec, tvec, camera_matrix, dist_coeffs, length=80):
    axis_3d = np.float32([[length,0,0],[0,length,0],[0,0,length]])
    nose_3d = np.float32([[0,0,0]])
    pts, _ = cv2.projectPoints(np.vstack([nose_3d, axis_3d]), rvec, tvec, camera_matrix, dist_coeffs)
    pts = pts.reshape(-1,2).astype(int)
    origin = tuple(pts[0])
    cv2.line(frame, origin, tuple(pts[1]), (0,0,255), 2)
    cv2.line(frame, origin, tuple(pts[2]), (0,255,0), 2)
    cv2.line(frame, origin, tuple(pts[3]), (255,0,0), 2)
    return frame

# --- Robust head pose (returns pitch,yaw,roll,reproj_err,rvec,tvec) ---
def get_head_pose(lm, frame_w, frame_h, focal_mult=1.2, reproj_thresh=12.0):
    try:
        idxs = [1, 152, 33, 263, 61, 291]  # verify with draw_landmark_indices if needed
        if any(i >= len(lm) for i in idxs):
            return None, None, None, None, None, None

        model_points = np.array([
            (0.0, 0.0, 0.0), (0.0, -330.0, -65.0),
            (-225.0, 170.0, -135.0), (225.0, 170.0, -135.0),
            (-150.0, -150.0, -125.0), (150.0, -150.0, -125.0)
        ], dtype=np.float64)

        image_points = np.array([
            (lm[1].x * frame_w, lm[1].y * frame_h),
            (lm[152].x * frame_w, lm[152].y * frame_h),
            (lm[33].x * frame_w, lm[33].y * frame_h),
            (lm[263].x * frame_w, lm[263].y * frame_h),
            (lm[61].x * frame_w, lm[61].y * frame_h),
            (lm[291].x * frame_w, lm[291].y * frame_h)
        ], dtype=np.float64)

        focal_length = frame_w * focal_mult
        center = (frame_w / 2.0, frame_h / 2.0)
        camera_matrix = np.array([[focal_length,0,center[0]],[0,focal_length,center[1]],[0,0,1]], dtype=np.float64)
        dist_coeffs = np.zeros((4,1), dtype=np.float64)

        success, rvec, tvec, inliers = cv2.solvePnPRansac(
            model_points, image_points, camera_matrix, dist_coeffs,
            flags=cv2.SOLVEPNP_EPNP, reprojectionError=8.0, iterationsCount=100
        )

        if not success:
            success2, rvec, tvec = cv2.solvePnP(model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)
            if not success2:
                return None, None, None, None, None, None

        reproj_err = _reprojection_error(model_points, image_points, rvec, tvec, camera_matrix, dist_coeffs)
        if reproj_err > reproj_thresh:
            return None, None, None, reproj_err, None, None

        rmat, _ = cv2.Rodrigues(rvec)
        sy = math.sqrt(rmat[0,0]**2 + rmat[1,0]**2)
        singular = sy < 1e-6
        if not singular:
            x = math.atan2(rmat[2,1], rmat[2,2])
            y = math.atan2(-rmat[2,0], sy)
            z = math.atan2(rmat[1,0], rmat[0,0])
        else:
            x = math.atan2(-rmat[1,2], rmat[1,1])
            y = math.atan2(-rmat[2,0], sy)
            z = 0.0

        pitch = math.degrees(x)
        yaw = math.degrees(y)
        roll = math.degrees(z)
        return pitch, yaw, roll, float(reproj_err), rvec, tvec

    except Exception:
        return None, None, None, None, None, None

# --- 3-class head classifier (yaw only) ---
def headpose_class_from_yaw(yaw, yaw_thresh=15):
    if yaw is None:
        return "UNKNOWN"
    if abs(yaw) <= yaw_thresh:
        return "CENTER"
    elif yaw < -yaw_thresh:
        return "RIGHT"
    else:
        return "LEFT"


In [ ]:
# -----------------------------
# Compute frame segments for ground truth
# -----------------------------
cap_tmp = cv2.VideoCapture(VIDEO_PATH)
if not cap_tmp.isOpened():
    raise ValueError(f"Video not found: {VIDEO_PATH}")
fps = cap_tmp.get(cv2.CAP_PROP_FPS) or 20.0
total_frames = int(cap_tmp.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
cap_tmp.release()

frames_per_segment = int(3 * fps)
left_segment = range(0, frames_per_segment)
center_segment = range(frames_per_segment, 2*frames_per_segment)
right_segment = range(2*frames_per_segment, 3*frames_per_segment)

In [ ]:
# -----------------------------
# Calibration per-eye (percentile-based)
# -----------------------------
def calibrate_per_eye(video_path, sample_every_n=1):
    cap = cv2.VideoCapture(video_path)
    left_ratios, right_ratios = [], []
    idx = 0
    with mp.solutions.face_mesh.FaceMesh(static_image_mode=False, refine_landmarks=True,
                                         max_num_faces=1, min_detection_confidence=0.6,
                                         min_tracking_confidence=0.6) as face:
        while True:
            ret, frame = cap.read()
            if not ret: break
            if sample_every_n>1 and (idx % sample_every_n)!=0:
                idx += 1
                continue
            fh, fw = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = face.process(rgb)
            if not res.multi_face_landmarks:
                idx += 1
                continue
            lm = res.multi_face_landmarks[0].landmark
            left_eye_img, right_eye_img, (left_box, right_box) = crop_both_eyes(frame)
            if left_box and left_eye_img is not None and left_eye_img.size>0:
                try:
                    _, (rx, _) = iris_centroid_and_ratio(lm, LEFT_IRIS_LMS, left_box, fw, fh)
                    if 0 <= rx <= 1: left_ratios.append(rx)
                except:
                    pass
            if right_box and right_eye_img is not None and right_eye_img.size>0:
                try:
                    _, (rx, _) = iris_centroid_and_ratio(lm, RIGHT_IRIS_LMS, right_box, fw, fh)
                    if 0 <= rx <= 1: right_ratios.append(rx)
                except:
                    pass
            idx += 1
    cap.release()

    def stats(arr):
        if len(arr)==0:
            return 0.0, 1.0
        lo = float(np.percentile(arr, 5))
        hi = float(np.percentile(arr, 95))
        # add a tiny epsilon to avoid degenerate intervals
        if hi <= lo:
            hi = lo + 1e-3
        return lo, hi

    lmin, lmax = stats(left_ratios)
    rmin, rmax = stats(right_ratios)
    return (lmin, lmax), (rmin, rmax)

print("Calibrating per-eye (this may take a few seconds)...")
(left_min,left_max), (right_min,right_max) = calibrate_per_eye(VIDEO_PATH, sample_every_n=1)
print("Calibration L:", left_min, left_max, " R:", right_min, right_max)

Calibrating per-eye (this may take a few seconds)...
Calibration L: 0.3432900292348926 0.6711075691292189  R: 0.33019000482116495 0.6851919537531793


In [ ]:
# -----------------------------
# Decision from ratio
# -----------------------------
def gaze_from_ratio(ratio, min_r, max_r, center_margin=0.30):
    if ratio is None or max_r <= min_r:
        return "UNKNOWN"
    rel = (ratio - min_r) / (max_r - min_r)
    # clamp
    rel = max(0.0, min(1.0, rel))
    left_thresh = 0.5 - center_margin/2
    right_thresh = 0.5 + center_margin/2
    if rel < left_thresh:
        return "LEFT"
    elif rel > right_thresh:
        return "RIGHT"
    else:
      return "CENTER"

In [ ]:
from collections import deque

# -----------------------------
# Main evaluation loop: testing-only, medium rules
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError("Cannot open video")

preds = []
records = []
frame_idx = 0
left_ema = EMA(alpha=0.6)
right_ema = EMA(alpha=0.6)
head_ema = EMA(alpha=0.6)
prev_head_class = "CENTER"  # smoothing for yaw

# --- SETTINGS ---
process_every_sec = 1.0
frame_skip = int(process_every_sec * fps)  # process every 1 second
att_window_gaze = deque(maxlen=int(4 * fps / frame_skip))   # gaze off >= 4 sec
att_window_head = deque(maxlen=int(6 * fps / frame_skip))   # head turned >= 6 sec

# --- OUTPUT VIDEO SETUP ---
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_video_path = os.path.join(OUT_DIR, "gaze_head_debug.mp4")
fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_video = cv2.VideoWriter(out_video_path, fourcc, fps, (fw, fh))

# --- last known values for overlay ---
last_pred = "CENTER"
last_head_class = "CENTER"
last_att = True

with mp.solutions.face_mesh.FaceMesh(static_image_mode=False, refine_landmarks=True,
                                     max_num_faces=1, min_detection_confidence=0.6,
                                     min_tracking_confidence=0.6) as face:
    pbar = tqdm(total=total_frames, desc="Processing frames")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        fh, fw = frame.shape[:2]

        pred = last_pred
        head_class = last_head_class
        attentive = last_att
        reason = ""

        # --- PROCESS EVERY N FRAMES ---
        if frame_idx % frame_skip == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = face.process(rgb)
            left_vote = right_vote = None

            if res.multi_face_landmarks:
                lm = res.multi_face_landmarks[0].landmark
                left_eye_img, right_eye_img, (left_box, right_box) = crop_both_eyes(frame)

                # LEFT eye
                if left_box and left_eye_img is not None and left_eye_img.size>0:
                    score = eye_open_score(lm, LEFT_EYE_LMS, fh)
                    blink_thresh = max(3.0, fh * 0.005)
                    if score > blink_thresh:
                        try:
                            _, (rx, _) = iris_centroid_and_ratio(lm, LEFT_IRIS_LMS, left_box, fw, fh)
                            sm = left_ema.update(rx)
                            left_vote = gaze_from_ratio(sm, left_min, left_max, center_margin=0.30)
                        except:
                            left_vote = None

                # RIGHT eye
                if right_box and right_eye_img is not None and right_eye_img.size>0:
                    score = eye_open_score(lm, RIGHT_EYE_LMS, fh)
                    blink_thresh = max(3.0, fh * 0.005)
                    if score > blink_thresh:
                        try:
                            _, (rx, _) = iris_centroid_and_ratio(lm, RIGHT_IRIS_LMS, right_box, fw, fh)
                            sm = right_ema.update(rx)
                            right_vote = gaze_from_ratio(sm, right_min, right_max, center_margin=0.30)
                        except:
                            right_vote = None

                # Fuse eye votes
                votes = [v for v in (left_vote, right_vote) if v and v!="UNKNOWN"]
                if len(votes) == 0:
                    pred = "UNKNOWN"
                elif len(votes) == 1:
                    pred = votes[0]
                else:
                    if left_vote == right_vote:
                        pred = left_vote
                    elif "CENTER" in votes:
                        pred = "CENTER"
                    else:
                        pred = left_vote

                # HEAD POSE: safe call now that lm exists
                pitch, yaw, roll, reproj_err, rvec, tvec = get_head_pose(lm, fw, fh, focal_mult=1.2, reproj_thresh=12.0)
                if yaw is None or reproj_err is None:
                    head_class = "UNKNOWN"
                else:
                    sm_yaw = head_ema.update(yaw)
                    head_class = headpose_class_from_yaw(sm_yaw, yaw_thresh=15)
                    prev_head_class = head_class

            else:
                # no face detected
                lm = None
                pred = "UNKNOWN"
                head_class = "UNKNOWN"


            # --- ATTENTIVENESS SCORING ---
            att_window_gaze.append(pred != "CENTER")
            att_window_head.append(head_class != "CENTER")

            attentive = True
            if sum(att_window_gaze) >= att_window_gaze.maxlen:
                attentive = False
                reason = "Gaze away >= 4s"
            elif sum(att_window_head) >= att_window_head.maxlen:
                attentive = False
                reason = "Head turned >= 6s"

            # update last known values for overlay
            last_pred = pred
            last_head_class = head_class
            last_att = attentive

        # --- RECORD ---
        records.append({
            "frame": frame_idx,
            "timestamp_sec": frame_idx / fps,
            "gaze": last_pred,
            "head_class": last_head_class,
            "attentive": last_att,
            "reason": reason
        })

        # --- VISUALIZATION ---
        overlay = frame.copy()
        cv2.putText(overlay, f"Gaze: {last_pred}", (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
        cv2.putText(overlay, f"Head: {last_head_class}", (20,80), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)
        att_text = "Attentive" if last_att else "Distracted"
        color = (0,255,0) if last_att else (0,0,255)
        cv2.putText(overlay, att_text, (fw-220, fh-30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
        out_video.write(overlay)

        frame_idx += 1
        pbar.update(1)
    pbar.close()

cap.release()
out_video.release()

# --- CLEAN SUMMARY CSV ---
summary_df = pd.DataFrame(records)
summary_csv_path = os.path.join(OUT_DIR, "engagement_summary.csv")
summary_df.to_csv(summary_csv_path, index=False)

# --- ATTENTIVENESS SCORE OVER VIDEO ---
total_frames_processed = len(records)
attentive_frames = sum([1 for r in records if r['attentive']])
att_score = (attentive_frames / total_frames_processed) * 100

# --- INTERPRETATION ---
if att_score >= 85:
    interpretation = "Highly attentive"
elif att_score >= 70:
    interpretation = "Moderately attentive"
else:
    interpretation = "Low attentiveness / distracted"

print(f"Attentiveness Score: {att_score:.2f}%")
print(f"Interpretation: {interpretation}")

# Save score & interpretation
summary_df_summary = pd.DataFrame([{
    "score": att_score,
    "interpretation": interpretation
}])
summary_df_summary.to_csv(os.path.join(OUT_DIR, "engagement_score.csv"), index=False)
print("Saved engagement score CSV:", os.path.join(OUT_DIR, "engagement_score.csv"))
print("Saved annotated video:", out_video_path)


Processing frames: 100%|██████████| 1525/1525 [00:29<00:00, 51.30it/s]

Attentiveness Score: 54.75%
Interpretation: Low attentiveness / distracted
Saved engagement score CSV: /content/gaze_debug/engagement_score.csv
Saved annotated video: /content/gaze_debug/gaze_head_debug.mp4
